# EEBG2026 HyPhy Selection Tutorial - Google Colab

This notebook runs the workshop in Google Colab. It clones the repository, installs HyPhy with `apt`, installs Python plotting/simulation packages with `pip`, verifies the bundled empirical data in `data/22-empirical`, runs the HyPhy analyses, and collects tables/figures.


## 1. Clone The Workshop Repository

Run this first. If you are editing a fork, change `REPO_URL` before running the cell.

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/aglucaci/selection-tutorial.git"
WORKDIR = Path("/content/selection-tutorial")

if not WORKDIR.exists():
    !git clone {REPO_URL} {WORKDIR}
else:
    print(f"Using existing checkout: {WORKDIR}")

os.chdir(WORKDIR)
print(Path.cwd())

## 2. Install HyPhy And Python Dependencies

This can take a few minutes on a fresh Colab runtime.

In [ ]:
!bash scripts/00_setup_colab.sh

## 3. Verify Bundled Empirical Data


In [ ]:
!bash scripts/01_download_tutorial_data.sh
!find data/22-empirical -maxdepth 1 -type f | sort | head -40

from pathlib import Path
import re
import pandas as pd
from IPython.display import display

empirical_dir = Path('data/22-empirical')
empirical_files = sorted(
    path for path in empirical_dir.iterdir()
    if path.suffix in {'.nex', '.mtnex'} or path.name.endswith('.masked_nex')
)

def parse_nexus_matrix(path):
    text = path.read_text(encoding='utf-8', errors='replace')
    in_matrix = False
    records = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        upper = line.upper()
        if upper.startswith('MATRIX'):
            in_matrix = True
            remainder = line[6:].strip()
            if not remainder:
                continue
            line = remainder
        if not in_matrix:
            continue
        if line.startswith(';') or upper.startswith('END;'):
            break
        if line.startswith('['):
            continue
        line = line.rstrip(';').strip()
        match = re.match(r"^'([^']+)'\s+([A-Za-z?\-]+)$", line)
        if not match:
            match = re.match(r'^(\S+)\s+([A-Za-z?\-]+)$', line)
        if match:
            records.append((match.group(1), match.group(2).upper()))
    return records, text

rows = []
for path in empirical_files:
    records, text = parse_nexus_matrix(path)
    sequences = [seq for _, seq in records]
    nt_sites = max((len(seq) for seq in sequences), default=0)
    total_chars = sum(len(seq) for seq in sequences)
    acgt = sum(seq.count(base) for seq in sequences for base in 'ACGT')
    gc = sum(seq.count(base) for seq in sequences for base in 'GC')
    gaps = sum(seq.count('-') for seq in sequences)
    ambiguous = sum(sum(1 for char in seq if char not in 'ACGT-') for seq in sequences)
    rows.append({
        'dataset': path.name,
        'sequences': len(sequences),
        'nt sites': nt_sites,
        'codon sites': nt_sites // 3 if nt_sites and nt_sites % 3 == 0 else None,
        'GC %': round(100 * gc / acgt, 1) if acgt else None,
        'gap %': round(100 * gaps / total_chars, 1) if total_chars else None,
        'ambiguous %': round(100 * ambiguous / total_chars, 1) if total_chars else None,
        'tree?': 'yes' if 'BEGIN TREES' in text.upper() else 'no',
        'size KB': round(path.stat().st_size / 1024, 1),
    })

empirical_summary = pd.DataFrame(rows)
display(empirical_summary)


In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

empirical_dir = Path('data/22-empirical')
empirical_files = sorted(
    path for path in empirical_dir.iterdir()
    if path.suffix in {'.nex', '.mtnex'} or path.name.endswith('.masked_nex')
)

if not empirical_files:
    raise FileNotFoundError('No empirical Nexus files found in data/22-empirical')

options = [(path.name, str(path)) for path in empirical_files]
default = next((str(path) for path in empirical_files if path.name == 'HIVvif.nex'), str(empirical_files[0]))

session1_gene = widgets.Dropdown(
    options=options,
    value=default,
    description='Gene dataset:',
    layout=widgets.Layout(width='70%'),
    style={'description_width': 'initial'},
)

display(session1_gene)


## 4. Session 1 - Gene-Wide Selection

In this session you will run BUSTED, a gene-wide test for episodic diversifying selection. BUSTED asks whether at least one codon site on the tested branches has experienced positive selection at some point in the tree. A significant result is evidence for selection somewhere in the gene, but it does not identify the exact codon site or tell you which biological episode caused it.

You will run three versions of the same gene-wide question:

- Standard BUSTED: the baseline gene-wide test.
- BUSTED with synonymous-rate variation: allows synonymous rates to vary across sites, which can matter when mutation or constraint is uneven.
- BUSTED with multiple-hit modeling: allows double and triple nucleotide changes, which can reduce misleading signal in some alignments.

Choose one empirical dataset from `data/22-empirical` with the dropdown above. Small datasets are better for a quick Colab run; larger datasets may take longer.

Prompts:

- Before looking at the p-values, what kind of biological scenario would make this gene a good candidate for episodic diversifying selection?
- What biological question is BUSTED answering for the gene you selected?
- Does a significant BUSTED result identify the selected codon site? What would you need to run next to localize the signal?
- Did synonymous-rate variation change the result enough to affect your interpretation?
- Did multi-hit modeling change the result enough to affect your interpretation?
- Which result would you report in a manuscript, and what caveat would you include?


In [ ]:
import os

selected_alignment = session1_gene.value
os.environ['SESSION1_ALIGNMENT'] = selected_alignment
print(f'Running Session 1 on: {selected_alignment}')

!bash scripts/02_run_gene_wide.sh
!python scripts/07_collect_results.py


In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv('tables/session1_gene_wide_summary_long.csv'))
display(Image('figures/session1_gene_wide_pvalues.png'))

## 5. Session 2 - Site-Level Selection

In this session you will move from a gene-wide test to codon-level evidence. FEL, MEME, and FUBAR all summarize selection by site, but they are not asking identical questions.

FEL looks for pervasive selection, meaning a site has an elevated or reduced nonsynonymous rate across the branches being tested. MEME looks for episodic diversifying selection, meaning a site may have experienced positive selection on only a subset of branches. FUBAR is a fast Bayesian method that can be useful for scanning patterns across sites.

Treat the site table as a map of candidate codons, not as final biological proof. Sites can be sensitive to alignment quality, recombination, model assumptions, and the number of sequences in the dataset.

Prompts:

- What is the difference between pervasive selection and episodic diversifying selection?
- Which codons were detected by FEL, and do they suggest purifying or diversifying selection?
- Which codons were detected by MEME, and are they the same codons FEL found?
- Where do FEL and MEME disagree, and what biological or statistical explanation could account for the difference?
- If you had a protein structure or domain map, which sites would you annotate first?
- What would you check in the alignment before trusting a surprising site-level result?


In [ ]:
!bash scripts/03_run_site_level.sh
!python scripts/07_collect_results.py

In [ ]:
sites = pd.read_csv('tables/session2_site_level_tables.csv')
display(sites.head())
for figure in sorted(Path('figures').glob('session2_site_level_*.png')):
    display(Image(str(figure)))

## 6. Session 3 - Branch-Level Selection

This session asks whether particular branches, rather than the whole tree, show evidence of episodic diversifying selection. aBSREL fits branch-specific models and then tests branches individually, so the result is a branch-level scan rather than a gene-wide yes-or-no answer.

Because many branches are tested, multiple-testing correction matters. An uncorrected low p-value can look exciting, but the corrected result is the one you should use when deciding which branches have convincing evidence.

The empirical files used here do not define named test and reference branch sets, so this session focuses on aBSREL rather than RELAX. If you later add branch labels, you could ask different lineage-comparison questions.

Prompts:

- What does aBSREL test that BUSTED does not?
- How many branches were tested, and why does that make multiple-testing correction necessary?
- Which branches, if any, remain significant after correction?
- Are significant branches terminal branches, internal branches, or a mixture? How does that affect interpretation?
- What biological story could explain selection on those branches?
- How would explicit branch labels change the questions you could ask with this dataset?


In [ ]:
!bash scripts/04_run_branch_lineage.sh
!python scripts/07_collect_results.py

In [ ]:
display(pd.read_csv('tables/session3_branch_lineage_summary_long.csv'))
if Path('figures/session3_branch_lineage_pvalues.png').exists():
    display(Image('figures/session3_branch_lineage_pvalues.png'))

## 7. Session 4 - Simulation And Model Checking

This session gives you a known truth table, then asks whether the HyPhy methods recover it. The simulated alignment has blocks with different omega regimes, so you can compare inferred evidence against the conditions used to generate the data.

Use this session to separate three ideas that often get mixed together: false positives, limited power, and model mismatch. A method can miss a true signal if the dataset is short or the signal is weak. It can also report extra sites when the data violate assumptions or when there is random noise.

Prompts:

- Which simulated block should be easiest to detect, and why?
- Did the methods recover the positive-enriched region from the truth table?
- Were there signals outside the positive-enriched block? If so, would you call them false positives, noise, or something else?
- Which method seemed most conservative, and which seemed most sensitive?
- What is the difference between low power and a false positive in this exercise?
- How would more taxa, longer alignments, or stronger selection change what you expect to recover?


In [ ]:
!python scripts/05_simulate_codon_data.py
!bash scripts/06_run_simulated_selection.sh
!python scripts/07_collect_results.py

In [ ]:
display(pd.read_csv('tables/session4_simulated_summary_long.csv'))
for figure in [
    'figures/session4_simulation_truth_blocks.png',
    'figures/session4_simulated_truth_vs_detected.png',
]:
    if Path(figure).exists():
        display(Image(figure))

## 8. Download Results

Run this optional cell to bundle the generated `results/`, `tables/`, and `figures/` directories into one zip file in the Colab file browser.

In [ ]:
!zip -qr eebg2026_hyphy_outputs.zip results tables figures
print('Wrote /content/selection-tutorial/eebg2026_hyphy_outputs.zip')

## Mini-Report Template

Use this final prompt to turn the outputs into a short scientific interpretation. The goal is not to list every p-value. The goal is to state what the analyses suggest, how confident you are, and what would make the conclusion stronger.

- Dataset: Which empirical or simulated dataset did you analyze, and why is it biologically interesting?
- Gene-wide result: Did BUSTED support episodic diversifying selection? Which model variant would you emphasize?
- Site-level result: Which codons are the strongest candidates, and do different methods agree?
- Branch-level result: Did aBSREL identify any branches after correction? What might those branches represent?
- Biological interpretation: What is the most plausible selection story supported by these results?
- Caveats: What alignment, sampling, model, or power issues could affect the conclusion?
- Next steps: What analysis, visualization, validation, or additional data would you want next?
